# 03 - Feature Engineering and Pre-Processing

##### 03-00 General Model Plan

Feature Engineering

- Create `duration_days` from the cleaned date columns
- Remove invalid and outlier duration rows
- Create the target variable from duration ranges
- Save the final cleaned modeling dataset

Pre-Processing

- Split dataset into training and testing subsets
- Split pre-processing steps into 3 groups (Text, Categories, and Words/Lengths)
- Use TF-IDF vectorization to convert text values (summaries and descriptions) to numeric matrices
- Use One Hot Encoder to encode catgories into numeric values with no relations, e.g, Bug and Improvement are completely unrelated
- Use Standard Scaler to standardizes the numeric values

Model training

- Use Logistic Regression


## 03-01 Importing CSV file and defining the target

Before we start training the model using the cleaned dataset, we need to create the target value from the cleaned date columns.

In [40]:
from pathlib import Path
import pandas as pd

jira_df = pd.read_csv("../data/processed/jira_issues_cleaned.csv", index_col=0)

task_df = jira_df.copy()

for col in ["created", "resolutiondate"]:
    task_df[col] = pd.to_datetime(task_df[col])

task_df.head()

,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,49,9,176,28,2005-01-01 07:47:50,2005-01-01 07:50:46
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,2004-12-25 22:50:30,2004-12-30 05:30:36
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,2004-12-27 01:34:17,2005-01-03 10:21:28
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,2004-12-28 18:50:52,2005-01-03 11:29:27
7,Port changes from tomcat-catalina,Since the [naming] sources were extracted from...,Major,Bug,33,4,1060,135,2005-01-03 11:16:07,2005-01-03 11:38:03


## 03-02 Create duration days derived variable

Calculate task duration in days.

This target is derived from:

`duration_days = resolutiondate - created`

In [41]:
task_df["duration_days"] = (task_df["resolutiondate"] - task_df["created"]).dt.total_seconds() / (60 * 60 * 24)

task_df["duration_days"].describe()

count    743631.000000
mean        123.703169
std         358.280600
min           0.000000
25%           0.963420
50%           8.163681
75%          61.880231
max        8001.517257
Name: duration_days, dtype: float64

The above statistics show target-distribution issues that need to be handled before modeling:

- The target has a strong long-tail pattern.
- Extreme outliers pull the mean far above the median.

#### Inspect long-tail durations

Before removing long-running tasks, first inspect how many rows would be affected.

In [42]:
display(task_df["duration_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

durations_over_120_days = (task_df["duration_days"] > 120).sum()
print(f"Rows with duration_days > 120: {durations_over_120_days:,}")

count    743631.000000
mean        123.703169
std         358.280600
min           0.000000
50%           8.163681
75%          61.880231
90%         321.861343
95%         699.164010
99%        1811.827084
max        8001.517257
Name: duration_days, dtype: float64

Rows with duration_days > 120: 134,559


Removing this many rows is a significant change, although the dataset may still contain enough records for modeling.

Tasks that exceed 90 days may be signs of:

- stale issues
- abandoned tickets
- imported/migrated issues
- tickets closed administratively
- issues left open for months/years without active work
- old backlog cleanup

These rows may not represent normal task-completion behavior, so removing or capping them can make the target easier for the model to learn.

In [43]:
task_df = task_df[task_df["duration_days"] <= 120].copy()

display(task_df.shape)
task_df.describe()

(609072, 11)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,609072.000000,609072.000000,6.090720e+05,609072.000000,609072,609072,609072.000000
mean,56.786079,7.854341,7.724587e+02,71.337853,2015-12-05 16:34:07.259068,2015-12-22 04:57:17.258643,16.516088
min,0.000000,0.000000,0.000000e+00,0.000000,2002-05-10 18:40:29,2002-05-10 18:54:17,0.000000
25%,40.000000,5.000000,1.070000e+02,15.000000,2012-08-29 22:37:44,2012-09-12 15:02:07.250000,0.624326
50%,54.000000,7.000000,2.650000e+02,37.000000,2016-03-06 08:19:13.500000,2016-03-22 13:01:21.500000,4.448709
75%,70.000000,10.000000,6.150000e+02,78.000000,2019-10-01 16:16:42.500000,2019-10-20 00:38:11,20.209598
max,255.000000,46.000000,1.602994e+06,139288.000000,2024-11-06 14:09:26,2025-02-26 18:17:12,119.999815
std,23.449556,3.582614,4.638295e+03,308.752036,NaN,NaN,25.862097


Tasks resolved in under 2 hours may represent very small fixes, administrative updates, or tickets closed immediately after creation. Removing them can reduce short-duration skew, but this should be treated as a modeling decision rather than a universal rule.

In [44]:
task_df = task_df[task_df["duration_days"] >= (2/24)].copy()

## 03-03 Create duration classification target

Create a classification target from `duration_days`.

The goal is to choose ranges that are:

- simple enough to explain
- not overly skewed toward one class
- aligned with what the task summaries and descriptions appear to represent

The earlier EDA showed two important cleanup decisions before classification:

- remove tasks completed in under 2 hours
- remove very long tasks above the selected long-tail cutoff

After those removals, compare a few possible class ranges before choosing the final classes.

### Duration ranges

- **Short:** same day and up to 3 days
- **Standard:** greater than 3 days and up to 3 weeks
- **Long-term:** greater than 3 weeks


In [45]:
def duration_category(days):
    if days <= 3:
        return "Short"
    if days <= 21:
        return "Standard"
    return "Long-running"

duration_order = ["Short", "Standard", "Long-running"]

task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_counts = task_df["duration_category"].value_counts().reindex(duration_order)
class_percentages = (task_df["duration_category"].value_counts(normalize=True).reindex(duration_order).mul(100).round(2))

duration_class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages,
})

display(task_df.shape)
display(task_df.describe())

duration_class_summary

(521571, 12)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,521571.000000,521571.000000,5.215710e+05,521571.000000,521571,521571,521571.000000
mean,57.023933,7.869448,8.294699e+02,76.188580,2016-02-06 02:05:46.780123,2016-02-25 08:53:39.865709,19.283253
min,1.000000,1.000000,0.000000e+00,0.000000,2002-05-10 18:40:29,2002-06-03 18:35:07,0.083333
25%,41.000000,5.000000,1.200000e+02,17.000000,2012-11-13 19:02:14,2012-12-04 22:29:58,1.486372
50%,54.000000,7.000000,2.880000e+02,40.000000,2016-06-04 00:17:48,2016-06-21 17:55:58,6.795150
75%,70.000000,10.000000,6.660000e+02,84.000000,2019-12-23 11:31:43,2020-01-15 08:02:18.500000,25.561007
max,255.000000,46.000000,1.602994e+06,139288.000000,2024-11-06 12:43:08,2025-02-26 18:17:12,119.999815
std,23.453111,3.584770,4.961862e+03,330.201376,NaN,NaN,26.976973


,count,percent
duration_category,,
Short,182204,34.93
Standard,191060,36.63
Long-running,148307,28.43


### 03-03.1 Spot-check task text by class

Sample a few summaries and descriptions from each final class. This is a manual sanity check that the classes are not obviously misleading for the type of task text the model will receive.

In [ ]:
sample_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "duration_category",
]

for category in duration_order:
    print(category)
    class_sample = task_df.loc[
        task_df["duration_category"] == category,
        sample_columns,
    ].sample(n=5, random_state=42)
    display(class_sample)

Short


,summary,description,priority_name,issuetype_name,duration_days,duration_category
951015,Axis2 integration - allow users to deploy jax-...,This patch contains the following updates to c...,Major,Improvement,0.406644,Short
818400,Performance improvement in OpenBitSetDISI.inPl...,NaN,Major,Improvement,1.588553,Short
963773,Fix incompatible API change on FsServerDefault...,From recently jdiff report: https://builds.apa...,Blocker,Bug,0.834039,Short
844041,[Release] Enable source packaging on CentOS,shasum is not available on CentOS and RHEL der...,Trivial,Improvement,0.088924,Short
992484,Add missing profiles in deploy_staging_jars.sh,The mvn build profiles for {{hudi-utilities-sl...,Blocker,Improvement,1.109965,Short


Standard


,summary,description,priority_name,issuetype_name,duration_days,duration_category
152922,TestAvroStorageUtils.testGetConcretePathFromGl...,I found that TestAvroStorageUtils.testGetConcr...,Major,Bug,6.863854,Standard
152042,NPE during GC with HierarchicalLedgerManager,"{noformat} 2013-02-11 14:06:28,904 - WARN - [G...",Minor,Bug,3.248877,Standard
285796,Enforce ErrorProne analysis in protobuf extens...,Java ErrorProne static analysis was [recently ...,P3,Improvement,13.855104,Standard
336573,Add left over data check to generated C parsers,Fix generated C parsers not checking for left ...,Minor,Improvement,6.194410,Standard
226991,Fixes examples of the official templates,First I will fix the following examples contai...,Minor,Sub-task,19.129572,Standard


Long-running


,summary,description,priority_name,issuetype_name,duration_days,duration_category
885879,`object of type 'NoneType' has no len()` error...,"Looks like in Python 3, webbrowser._tryorder c...",Normal,Bug,51.065856,Long-running
116935,TitleWindow contentBackgroundColor behavior is...,Steps to reproduce: 1.Run attached file 2.Use ...,Minor,Bug,52.064988,Long-running
496092,Findbugs HTML report link shows 0 warnings des...,Refer to Hadoop QA report below : https://issu...,Major,Sub-task,25.387986,Long-running
716748,[Broker-J] Upgrade hamcrest dependency to vers...,Upgrade hamcrest dependency to version 2.1,Major,Improvement,94.979248,Long-running
1145513,Pass node labels from shim to core.,Suggested by YUNIKORN-1362: The YuniKorn UI sh...,Major,Sub-task,23.401227,Long-running


The samples should not be expected to prove the classes are perfect. They only check that the ranges are reasonable enough for the first text-classification model. Actual performance should still be tested during model training.

## 03-04 Save final cleaned dataset

Save the final cleaned dataset after feature engineering.

We will also remove the `duration_days` at this stage to avoid leakage since the model is only going to be trained on the duration ranges for classification.

In [49]:
final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()
final_cleaned_df_sample = final_cleaned_df.sample(n=100, random_state=42)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_cleaned_path = output_dir / "final_cleaned.csv"
final_cleaned_df.to_csv(final_cleaned_path, index=False)

final_cleaned_df_sample_path = output_dir / "final_cleaned_sample.csv"
final_cleaned_df_sample.to_csv(final_cleaned_df_sample_path, index=False)

print(f"Final cleaned modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df.shape[1]:,}")
print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved final cleaned sample CSV file to: {final_cleaned_df_sample_path}")

final_cleaned_df.head()

Final cleaned modeling rows: 521,571
Final cleaned modeling columns: 9
Saved final cleaned CSV file to: ..\data\processed\final_cleaned.csv
Saved final cleaned sample CSV file to: ..\data\processed\final_cleaned_sample.csv


,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,duration_category
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,Standard
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,Standard
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,Standard
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,Major,Bug,40,7,1438,65,Long-running
9,ActionContext should always have a TextProvide...,Right now if you access the ActionContext befo...,Minor,Improvement,68,10,210,33,Short


In [9]:
print(f"Final cleaned modeling rows: {final_cleaned_df_sample.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df_sample.shape[1]:,}")

Final cleaned modeling rows: 100,000
Final cleaned modeling columns: 10
